# DuckDB and Parquet

DuckDB is an in-process analytical SQL database. It can query Parquet files directly without first loading them into a database table or a pandas DataFrame.

In this notebook, DuckDB will:

1. Read the MovieLens `movies.parquet` and `ratings.parquet` files.
2. Find movies rated by at least 100 users.
3. Keep movies with an average rating of 4.0 or higher.
4. Join the aggregates to movie titles.
5. Write the result to `popular-movies.parquet`.
6. Read the saved Parquet file and validate it.

No pandas API is used.

## Installation

```powershell
python -m pip install duckdb
```


## 1. Import DuckDB and define file locations

The database connection uses `:memory:`, so it does not create a DuckDB database file. The source and result data remain in Parquet.


In [ ]:
from pathlib import Path

import duckdb

PARQUET_DIR = Path(r"C:\data\movielens\parquet")
MOVIES_PATH = PARQUET_DIR / "movies.parquet"
RATINGS_PATH = PARQUET_DIR / "ratings.parquet"
POPULAR_MOVIES_PATH = PARQUET_DIR / "popular-movies.parquet"

for source_path in (MOVIES_PATH, RATINGS_PATH):
    if not source_path.is_file():
        raise FileNotFoundError(f"Required Parquet file not found: {source_path}")

connection = duckdb.connect(database=":memory:")

print(f"DuckDB version: {duckdb.__version__}")
print(f"Movies: {MOVIES_PATH}")
print(f"Ratings: {RATINGS_PATH}")
print(f"Output: {POPULAR_MOVIES_PATH}")


## 2. Small display helper

DuckDB returns tuples through `fetchall()`. This helper prints those native results with column headings and deliberately avoids pandas conversion methods.


In [ ]:
def show_query(sql, parameters=None, limit=None):
    result = connection.execute(sql, parameters or [])
    columns = [item[0] for item in result.description]
    rows = result.fetchmany(limit) if limit is not None else result.fetchall()
    print(" | ".join(columns))
    print("-" * 80)
    for row in rows:
        print(" | ".join(str(value) for value in row))
    return rows


def sql_string(path):
    # Return a safely quoted SQL string literal for a local path.
    return "'" + str(path).replace("'", "''") + "'"


## 3. Read Parquet files directly

`read_parquet()` is a DuckDB table function. DuckDB can push column selection and filters into the Parquet scan, reducing unnecessary I/O.

Here we inspect schemas and row counts without creating permanent database tables.


In [ ]:
movies_sql_path = sql_string(MOVIES_PATH)
ratings_sql_path = sql_string(RATINGS_PATH)

print("MOVIES SCHEMA")
show_query(f"DESCRIBE SELECT * FROM read_parquet({movies_sql_path})")

print("\nRATINGS SCHEMA")
show_query(f"DESCRIBE SELECT * FROM read_parquet({ratings_sql_path})")

movie_count = connection.execute(
    f"SELECT count(*) FROM read_parquet({movies_sql_path})"
).fetchone()[0]
rating_count = connection.execute(
    f"SELECT count(*) FROM read_parquet({ratings_sql_path})"
).fetchone()[0]

print(f"\nMovie rows: {movie_count:,}")
print(f"Rating rows: {rating_count:,}")


## 4. Preview movies and ratings

The query reads only the columns needed for the preview. `ORDER BY` makes the displayed result deterministic.


In [ ]:
print("MOVIES SAMPLE")
show_query(
    f"""
    SELECT movieId, title, genres
    FROM read_parquet({movies_sql_path})
    ORDER BY movieId
    LIMIT 5
    """
)

print("\nRATINGS SAMPLE")
show_query(
    f"""
    SELECT userId, movieId, rating, timestamp
    FROM read_parquet({ratings_sql_path})
    ORDER BY userId, movieId
    LIMIT 5
    """
)


## 5. Define the popular-movies query

The aggregation computes:

- `rating_count`: number of ratings received by a movie
- `distinct_user_count`: number of distinct users who rated it
- `average_rating`: mean score

The requirement is applied with `HAVING`: at least 100 distinct users and an average rating of at least 4.0. The result is joined to `movies.parquet` to add the title and genres.


In [ ]:
popular_movies_query = f"""
WITH rating_summary AS (
    SELECT
        movieId,
        count(*) AS rating_count,
        count(DISTINCT userId) AS distinct_user_count,
        avg(rating) AS average_rating
    FROM read_parquet({ratings_sql_path})
    GROUP BY movieId
    HAVING count(DISTINCT userId) >= 100
       AND avg(rating) >= 4.0
)
SELECT
    summary.movieId,
    movies.title,
    movies.genres,
    summary.rating_count,
    summary.distinct_user_count,
    round(summary.average_rating, 3) AS average_rating
FROM rating_summary AS summary
INNER JOIN read_parquet({movies_sql_path}) AS movies
    ON summary.movieId = movies.movieId
ORDER BY
    summary.average_rating DESC,
    summary.distinct_user_count DESC,
    summary.movieId
"""

popular_movies = connection.execute(popular_movies_query).fetchall()
popular_columns = [item[0] for item in connection.description]

print(" | ".join(popular_columns))
print("-" * 100)
for row in popular_movies:
    print(" | ".join(str(value) for value in row))

print(f"\nPopular movies found: {len(popular_movies)}")
assert popular_movies, "Expected at least one popular movie for the selected thresholds."


## 6. Write the query result to Parquet

DuckDB's `COPY` statement can write any query result directly to Parquet. ZSTD compression is selected for a compact analytical output.

The query is wrapped in parentheses because `COPY` expects a relation-producing statement.


In [ ]:
popular_movies_sql_path = sql_string(POPULAR_MOVIES_PATH)

connection.execute(
    f"""
    COPY ({popular_movies_query})
    TO {popular_movies_sql_path}
    (FORMAT PARQUET, COMPRESSION ZSTD)
    """
)

print(f"Created: {POPULAR_MOVIES_PATH}")
print(f"File size: {POPULAR_MOVIES_PATH.stat().st_size:,} bytes")


## 7. Read the saved result

The newly created file is queried directly. No import step or DataFrame is required.


In [ ]:
print("POPULAR MOVIES READ FROM PARQUET")
saved_rows = show_query(
    f"""
    SELECT
        movieId,
        title,
        genres,
        rating_count,
        distinct_user_count,
        average_rating
    FROM read_parquet({popular_movies_sql_path})
    ORDER BY average_rating DESC, distinct_user_count DESC, movieId
    """
)


## 8. Validate the output

The checks confirm that:

- every result has at least 100 distinct raters;
- every average rating is at least 4.0;
- the saved row count matches the query result;
- the output schema contains the expected columns.


In [ ]:
validation = connection.execute(
    f"""
    SELECT
        count(*) AS row_count,
        min(distinct_user_count) AS minimum_distinct_users,
        min(average_rating) AS minimum_average_rating
    FROM read_parquet({popular_movies_sql_path})
    """
).fetchone()

saved_schema = connection.execute(
    f"DESCRIBE SELECT * FROM read_parquet({popular_movies_sql_path})"
).fetchall()
saved_column_names = [row[0] for row in saved_schema]
expected_columns = [
    "movieId",
    "title",
    "genres",
    "rating_count",
    "distinct_user_count",
    "average_rating",
]

row_count, minimum_distinct_users, minimum_average_rating = validation

assert row_count == len(popular_movies)
assert minimum_distinct_users >= 100
assert minimum_average_rating >= 4.0
assert saved_column_names == expected_columns

print(f"Rows validated: {row_count}")
print(f"Minimum distinct users: {minimum_distinct_users}")
print(f"Minimum average rating: {minimum_average_rating}")
print("Output validation passed.")


## 9. Close the connection

Closing the in-memory connection releases its resources. The Parquet files remain available on disk.


In [ ]:
connection.close()
print("DuckDB connection closed.")


## Summary

- DuckDB queries Parquet files directly through `read_parquet()`.
- SQL aggregation, filtering, and joins do not require pandas.
- `COPY (query) TO ... (FORMAT PARQUET)` writes query results directly to Parquet.
- Column and filter pushdown allow DuckDB to avoid reading unnecessary Parquet data.
- The final dataset was written to:

`C:\data\movielens\parquet\popular-movies.parquet`
